# thinkCV — אימון YOLOv8 🧠  (v3 — נקי ויציב)

מקבלת את `thinkcv_dataset.zip` מהסטודיו → מאמנת → מחזירה **`best.onnx`** לטעינה בפלטפורמה.

גרסה זו מתקנת את כל מה שנתקלנו בו:
- נועלת `ultralytics==8.3.0` (8.4+ הסירו ייצוא ידידותי)
- מתקינה `onnxscript` מראש (החבילה שחסרה שוב ושוב)
- **בלי** tensorflow/tensorflowjs — הם גרמו להתנגשויות numpy ולא צריך אותם ל-ONNX
- נתיב האימון **דינמי** (`train`, `train2`… נבחר אוטומטית — לא קשיח)

**לפני הכל:** Runtime → Change runtime type → **GPU (T4)**. ואז ▶ תא-תא לפי הסדר.


### 1. התקנה  ⏱️ ~1 דק׳
רזה ויציב — רק מה שצריך ל-ONNX. אם בסוף תופיע אזהרת "Restart runtime" — התעלמו, ממשיכים.


In [ ]:
!pip install -q 'ultralytics==8.3.0' onnx onnxslim onnxscript onnxruntime
import ultralytics; ultralytics.checks()
from ultralytics import YOLO
print('\n✅ מוכן — המשיכו לתא הבא')


### 2. העלאת הדאטהסט
יפתח כפתור — בחרו את `thinkcv_dataset.zip`.


In [ ]:
import zipfile, os, glob, shutil
from google.colab import files
DATA_DIR='/content/dataset'
shutil.rmtree(DATA_DIR, ignore_errors=True); os.makedirs(DATA_DIR, exist_ok=True)
up=files.upload()  # ← thinkcv_dataset.zip
with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall(DATA_DIR)
n_img=len(glob.glob(DATA_DIR+'/images/*')); n_lbl=len(glob.glob(DATA_DIR+'/labels/*'))
print(f'נחלץ ✓  תמונות: {n_img} · תיוגים: {n_lbl}')
assert n_img>0, 'לא נמצאו תמונות — בדקו את ה-ZIP'


### 3. תיקון נתיב ב-data.yaml


In [ ]:
import yaml
yml_path=os.path.join(DATA_DIR,'data.yaml')
cfg=yaml.safe_load(open(yml_path))
cfg['path']=DATA_DIR; cfg['train']='images'; cfg['val']='images'
yaml.safe_dump(cfg, open(yml_path,'w'), allow_unicode=True)
print(open(yml_path).read())


### 4. אימון  ⏱️ כמה דקות
מעט תמונות → העלו `EPOCHS`. **חשוב:** דאטה מגוונת (זוויות/מרחקים/תאורות) = מודל טוב.
התא שומר את הנתיב המדויק של המודל אוטומטית ל-`best` — גם אם זה train2/train3.


In [ ]:
EPOCHS=100      # לבדיקה מהירה: 20
IMGSZ=640
model=YOLO('yolov8n.pt')
res=model.train(data=yml_path, epochs=EPOCHS, imgsz=IMGSZ, patience=30)
# נתיב דינמי — לא קשיח:
best=os.path.join(res.save_dir,'weights','best.pt')
print('\nהאימון הסתיים ✓  →', best)
assert os.path.exists(best), 'best.pt לא נמצא'


### 5. בדיקת המודל (מומלץ לפני הורדה)
מריץ את המודל על תמונת אימון — אם רואים זיהויים, המודל טוב.
מדפיס לכל קטגוריה את ה-mAP (דיוק): מעל ~0.8 = מצוין.


In [ ]:
m=YOLO(best)
test=glob.glob(DATA_DIR+'/images/*')[0]
r=m(test, conf=0.25, verbose=False)
print('זוהו בתמונת הבדיקה:', len(r[0].boxes), 'אובייקטים')
for b in r[0].boxes: print(f'  {m.names[int(b.cls)]}: {float(b.conf):.2f}')


### 6. הורדת `best.pt` (הקובץ הסופי, PyTorch)


In [ ]:
print('גודל:', round(os.path.getsize(best)/1e6,2),'MB')
files.download(best)


### 7. ייצוא ל-ONNX + הורדה  ✅
הקובץ שנטען בפלטפורמה תחת "מודל ONNX (best.onnx)". רץ מקומית בדפדפן.
**מקלידים בפלטפורמה את שמות הקטגוריות שמודפסות כאן, לפי הסדר.**


In [ ]:
onnx_path=YOLO(best).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print('נשמר:', onnx_path)
names=yaml.safe_load(open(yml_path)).get('names',[])
if isinstance(names,dict): names=[names[k] for k in sorted(names)]
print('\n📋 שמות קטגוריות (העתיקו לפלטפורמה):', ', '.join(map(str,names)))
files.download(str(onnx_path))


---
**סגירת הלולאה:** בפלטפורמה (הדף הראשי) → "מודל ONNX (best.onnx)" → בחרו `best.onnx` →
הדביקו שמות קטגוריות → "טען מודל" → "הפעל מצלמה". 🎉

אם תא כלשהו נכשל על numpy/import: **Runtime → Restart session** → הריצו שוב מתא 1.
